# Signatures, Metrics and a Training Set [Agent Patterns - Module 09]

> **MLCourse - Agentic AI - Agent Patterns**

An optimiser cannot improve anything without two things you must supply:

1. **Examples** - inputs paired with the output you actually wanted.
2. **A metric** - a Python function that scores a prediction against that
   wanted output and returns a number (or a bool).

That is the whole contract. This notebook builds both for a small, honest
task, and measures a **baseline** so notebook 03 has something to beat.

### What you will learn

1. Typed and constrained output fields (`Literal`).
2. `dspy.Example` and the `.with_inputs()` call everyone forgets.
3. Writing a metric - and why exact-match is the right choice here.
4. Running a baseline evaluation with 429-safe pacing.

### Key takeaways

- Examples encode *house conventions* the model cannot guess.
- A metric turns "sounds better" into a number you can defend.
- Always measure the baseline BEFORE you optimise. Otherwise you have no
  idea whether the optimiser helped or just cost you tokens.

### Setup: imports, environment, track discovery


In [ ]:
import os
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq)")
print(f"Key loaded : {bool(GROQ_API_KEY)}")


### Point DSPy at Groq (through LiteLLM)


In [ ]:
# dspy.LM is a thin wrapper over LiteLLM. The "groq/" prefix is the LiteLLM
# provider route - the rest is the Groq model id.

import dspy

lm = dspy.LM(
    f"groq/{MODEL}",
    api_key=GROQ_API_KEY,
    temperature=0.0,      # deterministic-ish: we are going to MEASURE things
    max_tokens=700,
    num_retries=5,        # LiteLLM backs off on 429 (Groq free tier = 8000 TPM)
)
dspy.configure(lm=lm)

print("DSPy configured.")
print("dspy version:", dspy.__version__)


### 1. The task: triage priority with an in-house rubric

A support desk assigns every incoming message a priority. The rubric is a
*company policy*, not general knowledge:

- **P0** - the customer cannot use the product at all (outage, locked out,
  crash on launch, data loss).
- **P1** - money is involved (wrong charge, refund, billing, payment
  failure), even if the product still works.
- **P2** - everything else: how-to questions, feature requests, feedback.

This is deliberately chosen. A general model has *no way* to know that
"charged twice" outranks "the export button is confusing" at this
particular company. That knowledge lives in the examples - which is
exactly the situation where prompt optimisation earns its keep.

### A typed signature: constrain the output space


In [ ]:
from typing import Literal

class Triage(dspy.Signature):
    """Assign a support priority to an incoming customer message."""

    message: str = dspy.InputField(desc="the raw customer message")
    priority: Literal["P0", "P1", "P2"] = dspy.OutputField(
        desc="the triage priority"
    )

print(Triage.__doc__)
print("input fields :", list(Triage.input_fields))
print("output fields:", list(Triage.output_fields))


Using `Literal["P0", "P1", "P2"]` instead of `str` matters: DSPy puts the
allowed values into the generated prompt *and* validates the parse. You get
fewer "Priority: high" style off-format answers for free.

Note what the signature still does **not** contain: the rubric itself.
That is on purpose - we want the optimiser to discover it from data.

### The data: 10 training examples, 12 held-out dev examples


In [ ]:
# Small on purpose: free-tier budget, and this is enough to show the effect. Each is (message, gold priority) under the house rubric above.

TRAIN = [
    ("I cannot log in at all, it says account locked since this morning.", "P0"),
    ("You charged my card twice for order 8812, please refund one.",       "P1"),
    ("How do I change the language in the settings screen?",               "P2"),
    ("The whole dashboard is blank for everyone on my team since 9am.",    "P0"),
    ("My subscription renewed at the old price, the invoice looks wrong.", "P1"),
    ("It would be nice if the export button remembered my last folder.",   "P2"),
    ("App crashes immediately on launch after the update, unusable.",      "P0"),
    ("Payment failed three times but the money left my account.",          "P1"),
    ("URGENT!! Add a dark mode toggle immediately, this is top priority!", "P2"),
    ("No hurry: the annual plan charged me the monthly rate by mistake.",  "P1"),
]

DEV = [
    ("Nobody in the office can open the site, it times out.",              "P0"),
    ("I was billed for two seats but we only ever had one.",               "P1"),
    ("Where can I find the keyboard shortcuts list?",                      "P2"),
    ("Everything I typed was lost when the editor froze and reset.",       "P0"),
    ("Please cancel my plan and refund the last charge.",                  "P1"),
    ("The dark theme colours are a bit low contrast, just feedback.",      "P2"),
    ("Login page returns a 500 error for all our users.",                  "P0"),
    ("An unexpected 49 rupee fee showed up on this month statement.",      "P1"),
    ("URGENT!!! You MUST add a bulk delete button, this is critical!!",    "P2"),
    ("No rush at all, but the invoice total is 200 more than quoted.",     "P1"),
    ("I am furious, the icons are ugly since the redesign.",               "P2"),
    ("Server returns 503 to every request, our shop is offline.",          "P0"),
]

# dspy.Example wraps a row. .with_inputs() tells DSPy which fields are the
# INPUTS - anything not listed is treated as the gold label. Forgetting this
# is the single most common DSPy beginner bug.
trainset = [dspy.Example(message=m, priority=p).with_inputs("message")
            for m, p in TRAIN]
devset = [dspy.Example(message=m, priority=p).with_inputs("message")
          for m, p in DEV]

print(f"trainset: {len(trainset)}  devset: {len(devset)}")
print("one example:", trainset[0])
print("its inputs :", trainset[0].inputs())


### 2. The metric

A DSPy metric is just a function `(example, prediction, trace=None) -> score`.
Here the label space is three fixed strings, so exact match is honest and
unambiguous. For free-text outputs you would need something fuzzier
(embedding similarity, or an LLM-as-judge - see module 07 of this track),
and every bit of fuzziness makes your before/after number less trustworthy.

### The metric


In [ ]:
def priority_match(example, pred, trace=None):
    """1.0 when the predicted priority equals the gold priority."""
    return float(str(pred.priority).strip().upper() == example.priority)

# Sanity-check the metric itself before trusting any score it produces.
class _Fake:
    def __init__(self, p): self.priority = p

print(priority_match(trainset[0], _Fake("P0")))   # expect 1.0
print(priority_match(trainset[0], _Fake("P2")))   # expect 0.0


### 3. A 429-safe evaluation loop

Groq's free tier is 8000 tokens per minute. DSPy ships `dspy.Evaluate`, but
we write the loop by hand here so you can see every moving part - and so we
can pace it. Rate-limit handling is not optional in a course notebook that
must run end to end.

### Evaluation with pacing and 429 backoff


In [ ]:
def evaluate(program, dataset, metric, pause=1.5, label=""):
    """Score `program` over `dataset`. Returns (accuracy, rows)."""
    rows, correct = [], 0.0
    for i, ex in enumerate(dataset, 1):
        for attempt in range(5):
            try:
                pred = program(**ex.inputs())
                break
            except Exception as e:                     # includes 429s
                wait = 2 ** attempt + random.random()
                print(f"  retry {attempt+1} in {wait:.1f}s ({type(e).__name__})")
                time.sleep(wait)
        else:
            raise RuntimeError("giving up after 5 retries")

        s = metric(ex, pred)
        correct += s
        rows.append((ex.message, ex.priority, str(pred.priority), s))
        print(f"  [{i}/{len(dataset)}] gold={ex.priority} pred={pred.priority} {'OK' if s else 'MISS'}")
        time.sleep(pause)

    acc = correct / len(dataset)
    print(f"\n{label} accuracy: {acc:.1%}  ({int(correct)}/{len(dataset)})")
    return acc, rows

print("evaluate() ready.")


### BASELINE: the un-optimised program


In [ ]:
# No examples, no rubric in the prompt. Just the signature.

baseline_program = dspy.Predict(Triage)

print("=== BASELINE (zero-shot, no demos) ===")
baseline_acc, baseline_rows = evaluate(
    baseline_program, devset, priority_match, label="Baseline"
)


### Where did it go wrong?


In [ ]:
print("Misclassified by the baseline:\n")
misses = [r for r in baseline_rows if r[3] == 0.0]
if not misses:
    print("  (none - the baseline already solved the dev set)")
for msg, gold, pred, _ in misses:
    print(f"  gold={gold} pred={pred}  {msg}")

print(f"\nBaseline accuracy to beat: {baseline_acc:.1%}")

# Persist it so notebook 03 compares against a real recorded number.
Path("baseline_score.txt").write_text(f"{baseline_acc:.4f}", encoding="utf-8")
print("saved -> baseline_score.txt")


### Reading the misses

Look at which rows failed. The typical pattern: the model applies *general*
intuition ("angry about money = urgent = P0", "feedback = ignorable") rather
than the house rubric it was never told. That gap - between general
plausibility and your organisation's actual convention - is precisely what
few-shot demonstrations fix.

### Pitfalls recap

- **Forgetting `.with_inputs()`** makes DSPy treat your gold label as an
  input and everything scores 100%. If a baseline looks suspiciously
  perfect, check this first.
- **Dev set leaking into the train set** inflates the after-number. Keep
  them disjoint, as above.
- **Tiny dev sets are noisy.** 12 rows means one flip = 8.3 points. Report
  the fraction (`6/8`), not just the percentage, so the reader can see the
  resolution.

### Next

Notebook 03 runs an optimiser over `trainset` and re-scores the same
`devset` with the same metric - a real, measured before/after.